In [ ]:
import numpy as np
import pandas as pd
from functools import partial, reduce
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor

In [2]:
df = pd.read_parquet("base_EPEX_df.parquet")
df.head()

,Incentive_forecast,Hour,FORECASTDEMAND,DCOILBRENTEU,sap_lag2d,Price (GBP/MWhe)
0,7589.5,2018-04-17 00:00:00+01:00,24160,73.18,1.7217,41.937710
1,7750.5,2018-04-17 01:00:00+01:00,21620,73.18,1.7217,39.910275
2,7924.0,2018-04-17 02:00:00+01:00,22210,73.18,1.7217,38.426365
3,8143.0,2018-04-17 03:00:00+01:00,22210,73.18,1.7217,31.947200
4,8207.5,2018-04-17 04:00:00+01:00,21000,73.18,1.7217,31.127599


In [3]:
# implements linear regression for each hour as its own model

target = "Price (GBP/MWhe)"

# --- Reshape from hourly rows to one row per day ---
d = df.sort_values("Hour").copy()
d["date"] = pd.to_datetime(d["Hour"].dt.date)   # naive local calendar date
d["hod"] = d["Hour"].dt.hour

# Keep only complete 24-hour days (drops DST days with 23/25 hours and partial days)
d = d[d.groupby("date")["Hour"].transform("size") == 24]

def wide(col, prefix):
    w = d.pivot(index="date", columns="hod", values=col)
    w.columns = [f"{prefix}_h{h:02d}" for h in w.columns]
    return w

demand = wide("FORECASTDEMAND", "demand")
wind = wide("Incentive_forecast", "wind")
oil = wide("DCOILBRENTEU", "oil")
gas = wide("sap_lag2d", "gas")
price = wide(target, "price")

# Autoregressive features: shift(freq="D") shifts by true calendar days,
# so missing days produce NaN rather than silently misaligned rows
price_lag1 = price.shift(1, freq="D").add_suffix("_lag1d")   # yesterday's 24 prices
price_lag7 = price.shift(7, freq="D").add_suffix("_lag7d")   # same weekday last week

X = (demand.join(wind, how="inner")
           .join(gas, how="inner")
           .join(oil, how="inner")
           .join(price_lag1, how="inner")
           .join(price_lag7, how="inner"))

X["day_of_week"] = X.index.dayofweek
X["month"] = X.index.month_name()
X = X.dropna()
Y = price.loc[X.index]    # 24 columns: the target vector for each day

In [ ]:
# --- Pipeline ---

numeric_features = [c for c in X.columns if c not in ["month", "day_of_week"]]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["day_of_week", "month"]),
])

baseline = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),   # multi-output: fits one linear model per hour
])

ridge = Pipeline([
    ("prep", preprocess),
    ("reg", Ridge()),   # multi-output: fits one linear model per hour
])

lgbm = Pipeline([
    ("prep", preprocess),
    ("reg", MultiOutputRegressor(
        LGBMRegressor(n_estimators=200, learning_rate=0.05, verbosity=-1)
    )),
])

# --- Time-ordered cross-validation ---
tscv = TimeSeriesSplit(n_splits=15)
rmses_base = []
rmses_ridge = []

for train_idx, test_idx in tscv.split(X):
    baseline.fit(X.iloc[train_idx], Y.iloc[train_idx])
    ridge.fit(X.iloc[train_idx], Y.iloc[train_idx])

    pred_base = baseline.predict(X.iloc[test_idx])
    pred_ridge = ridge.predict(X.iloc[test_idx])

    rmses_base.append(np.sqrt(np.mean((Y.iloc[test_idx].values - pred_base) ** 2)))
    rmses_ridge.append(np.sqrt(np.mean((Y.iloc[test_idx].values - pred_ridge) ** 2)))

print("CV RMSE per fold:", f"base: {np.round(rmses_base, 2)}, ridge: {np.round(rmses_ridge, 2)}",
      "mean:", f"base: {np.mean(rmses_base).round(2)}, ridge: {np.mean(rmses_ridge).round(2)}")

c:\Users\hamis\anaconda3\envs\epex\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\hamis\anaconda3\envs\epex\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


CV RMSE per fold: base: [15.49  8.08  8.2  10.59 33.93 89.71 61.4  56.4  16.1  17.93 14.92 31.3
 14.47 15.58 21.67], ridge: [ 7.92  7.08  6.82 10.49 34.09 89.33 58.66 54.59 14.55 16.76 14.24 30.55
 13.85 15.15 21.27] mean: base: 27.72, ridge: 26.36


In [5]:
# --- Hold-out evaluation (last 20% of days) ---
split = int(len(X) * 0.8)
baseline.fit(X.iloc[:split], Y.iloc[:split])
ridge.fit(X.iloc[:split], Y.iloc[:split])

pred_base = pd.DataFrame(baseline.predict(X.iloc[split:]), index=Y.index[split:], columns=Y.columns)
pred_ridge = pd.DataFrame(ridge.predict(X.iloc[split:]), index=Y.index[split:], columns=Y.columns)

err_base = pred_base - Y.iloc[split:]
err_ridge = pred_ridge - Y.iloc[split:]
print("Baseline MAE :", err_base.abs().values.mean())
print("Ridge MAE :", err_ridge.abs().values.mean())

print("Baseline RMSE:", np.sqrt((err_base ** 2).values.mean()))
print("Ridge RMSE:", np.sqrt((err_ridge ** 2).values.mean()))

print("MAE by hour (Ridge):\n", err_ridge.abs().mean().round(2))

Baseline MAE : 12.494294369266386
Ridge MAE : 11.857955283153757
Baseline RMSE: 18.246412542201487
Ridge RMSE: 17.3483511851196
MAE by hour (Ridge):
 price_h00     3.69
price_h01     4.58
price_h02     6.12
price_h03     7.40
price_h04     8.15
price_h05     9.62
price_h06    10.46
price_h07    10.75
price_h08    12.97
price_h09    13.60
price_h10    13.61
price_h11    14.10
price_h12    14.62
price_h13    14.56
price_h14    14.31
price_h15    13.48
price_h16    14.86
price_h17    21.77
price_h18    18.37
price_h19    15.61
price_h20    11.29
price_h21     9.32
price_h22     9.42
price_h23    11.96
dtype: float64


In [9]:
def metrics(y_true, y_pred):
    err = np.asarray(y_pred) - np.asarray(y_true)
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean())}

Y_test = Y.iloc[split:]
naive_prev_day = X.iloc[split:][[f"{c}_lag1d" for c in Y.columns]].values

results = pd.DataFrame({
    "Regression": metrics(Y_test, pred_base),
    "Ridge": metrics(Y_test, pred_ridge),
    "Prev day":   metrics(Y_test, naive_prev_day),
}).T
print(results.round(2))

# Per-hour MAE
by_hour = pd.DataFrame({
    "Regression": (pred_base - Y_test).abs().mean().values,
    "Ridge": (pred_ridge - Y_test).abs().mean().values,
    "Prev day":   np.abs(naive_prev_day - Y_test.values).mean(axis=0),
}, index=range(24))
print(by_hour.round(2))

# Improvement over baseline (positive = regression is better)
over_easy = 1 - results.loc["Ridge", "RMSE"] / results.loc["Prev day", "RMSE"]
over_base = 1 - results.loc["Ridge", "RMSE"] / results.loc["Regression", "RMSE"]

print(f"RMSE improvement of Ridge vs prev-day baseline: {over_easy:.1%}")
print(f"RMSE improvement of Ridge vs regression baseline: {over_base:.1%}")

              MAE   RMSE
Regression  12.49  18.25
Ridge       11.86  17.35
Prev day    17.49  28.50
    Regression  Ridge  Prev day
0         4.03   3.69     14.83
1         4.89   4.58     14.74
2         6.53   6.12     14.89
3         7.71   7.40     14.95
4         8.66   8.15     15.35
5        10.42   9.62     17.02
6        11.55  10.46     19.16
7        11.50  10.75     21.07
8        13.33  12.97     21.55
9        14.11  13.60     21.80
10       13.92  13.61     21.92
11       14.34  14.10     22.26
12       14.93  14.62     22.70
13       14.93  14.56     22.42
14       14.64  14.31     21.64
15       13.72  13.48     20.38
16       15.60  14.86     17.90
17       24.53  21.77     15.84
18       19.76  18.37     15.79
19       16.58  15.61     14.16
20       11.83  11.29     12.06
21        9.55   9.32     11.01
22       10.03   9.42     11.80
23       12.76  11.96     14.60
RMSE improvement of Ridge vs prev-day baseline: 39.1%
RMSE improvement of Ridge vs regression baseli